The script below generates a MIDI test file for one piano key, where the same note is played 127 times, each time with a different velocity from 1 to 127.

What it does:

1. Takes one command-line argument: <note>, i.e. a MIDI note number from 21 to 108 (the standard piano range).
2. Creates a MIDI file: note_reference_<note>.mid, e.g. for note 60, it creates note_reference_60.mid
    The MIDI file contains a MIDI sequence as follows:
    - Tempo is set to 120 BPM
    - The chosen note is repeated 127 times
    - Each repetition uses a different velocity (from 1 to 127)
    -Timing
        note_duration = 480 ticks (0.5 s)
        gap_duration = 480 ticks (0.5 s)
3. Saves the MIDI file
4. Prints a success message after saving

In [ ]:
from mido import MidiFile, MidiTrack, Message, MetaMessage

note = 69

if not 21 <= note <= 108:
    raise ValueError("Note must be between 21 and 108")

midi = MidiFile(ticks_per_beat=480)
piano_track = MidiTrack()
pedal_track = MidiTrack()

midi.tracks.append(piano_track)
midi.tracks.append(pedal_track)

piano_track.append(MetaMessage("set_tempo", tempo=500000, time=0))

for velocity in range(1, 128):
    piano_track.append(Message("note_on", note=note, velocity=velocity, time=480))
    piano_track.append(Message("note_off", note=note, velocity=0, time=480))

filename = f"note_reference_{note}.mid"
midi.save(filename)
print(f"Created {filename}")

The script below opens one MIDI file, extracts its note events, saves them to a text file, plots MIDI notes and statistics, previews the text in the notebook, and prints where the outputs were saved.

In [ ]:
import mido
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown


note = 69


midi_path = f"note_reference_{note}.mid"
text_path = f"note_reference_{note}.txt"
plot_path = f"note_reference_{note}.png"
title_text = midi_path


def midi_to_text_and_plot(
    midi_path,
    text_path,
    plot_path,
    title_text="MIDI Analysis",
    preview_lines=300
):
    midi = mido.MidiFile(midi_path)
    tempo = 500000
    tpq = midi.ticks_per_beat

    vel_vals, vel_times, notes, note_times = [], [], [], []

    with open(text_path, "w") as f:
        f.write(f"MIDI File: {midi_path}\n")
        f.write(f"Ticks per Beat: {tpq}\n\n")

        for i, track in enumerate(midi.tracks):
            f.write(f"Track {i}: {track.name}\n")
            f.write("-" * 40 + "\n")
            ticks = 0

            for msg in track:
                ticks += msg.time

                if msg.type == "set_tempo":
                    tempo = msg.tempo

                sec = mido.tick2second(ticks, tpq, tempo)

                if msg.type == "note_on" and msg.velocity > 0 and msg.note==note:
                    vel_vals.append(msg.velocity)
                    vel_times.append(sec)
                    notes.append(msg.note)
                    note_times.append(sec)

                    f.write(
                        f"Time {sec:.3f} sec | NOTE_ON  | "
                        f"Note: {msg.note:3d} | Velocity: {msg.velocity:3d}\n"
                    )

                elif msg.type == "note_off" or (msg.type == "note_on" and msg.velocity == 0):
                    f.write(
                        f"Time {sec:.3f} sec | NOTE_OFF | "
                        f"Note: {msg.note:3d} | Velocity: 0\n"
                    )

            f.write("\n")

    if vel_vals:
        fig, ax = plt.subplots(2, 2, figsize=(11, 7.5))

        ax[0, 0].scatter(vel_times, vel_vals, alpha=0.7)
        ax[0, 0].set_title("Velocity vs. Time")
        ax[0, 0].set_xlabel("Time (seconds)")
        ax[0, 0].set_ylabel("Velocity (0-127)")
        ax[0, 0].set_ylim(0, 127)
        ax[0, 0].set_xlim(0, 127)
        ax[0, 0].grid()
        ax[0, 0].text(-0.15, 1.12, title_text, transform=ax[0, 0].transAxes, fontsize=12)

        ax[1, 0].scatter(note_times, notes, color="purple", alpha=0.7)
        ax[1, 0].set_title("MIDI Note vs. Time")
        ax[1, 0].set_xlabel("Time (seconds)")
        ax[1, 0].set_ylabel("MIDI Note (21-108)")
        ax[1, 0].set_ylim(21, 108)
        ax[1, 0].grid()

        ax[0, 1].hist(
            vel_vals, bins=128, range=(0, 127),
            color="green", alpha=0.7, edgecolor="black"
        )
        ax[0, 1].set_title("Histogram of Velocity Values")
        ax[0, 1].set_xlabel("Velocity")
        ax[0, 1].set_ylabel("Count")
        ax[0, 1].set_xlim(0, 127)
        ax[0, 1].grid()

        ax[1, 1].hist(
            notes, bins=88, range=(21, 108),
            color="orange", alpha=0.7, edgecolor="black"
        )
        ax[1, 1].set_title("Histogram of MIDI Notes")
        ax[1, 1].set_xlabel("MIDI Note (21-108)")
        ax[1, 1].set_ylabel("Count")
        ax[1, 1].set_xlim(21, 108)
        ax[1, 1].grid()

        plt.tight_layout()
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        #display(Markdown("### Saved PNG preview"))
        #display(Image(filename=plot_path))
    else:
        display(Markdown("**No NOTE_ON events with velocity > 0 were found.**"))

    display(Markdown(f"### Text preview: `{text_path}`"))
    with open(text_path, "r") as f:
        lines = f.readlines()

    preview = "".join(lines[:preview_lines])
    if len(lines) > preview_lines:
        preview += "\n... (truncated) ..."

    display(Markdown(f"```text\n{preview}\n```"))

    print(f"PLOTS saved to: {plot_path}")
    print(f"MIDI events saved to: {text_path}")




midi_to_text_and_plot(
    midi_path=midi_path,
    text_path=text_path,
    plot_path=plot_path,
    title_text=title_text,
    preview_lines=300
)

The script below turns a MIDI file into a WAV audio file using FluidSynth, then displays an audio player in Jupyter to play generated audio.

What it does:

1. uses FluidSynth
2. loads the soundfont SteinGP.sf2
3. reads the MIDI file note_reference_69.mid
4. uses gain 1
5. renders the sound directly to note_reference_69.wav

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Audio


note = 69

input_midi = f"note_reference_{note}.mid"
output_wav = f"note_reference_{note}.wav"
soundfont = "SOUNDFONTS/MuseScore.sf2"

subprocess.run([
    "SOUNDFONTS/bin/fluidsynth.exe",
    "-ni",
    soundfont,
    input_midi,
    "-g",
    "1",
    #"--verbose",
    "-F",
    output_wav,
], check=True)

Audio(filename=output_wav)

The script below takes a WAV audio file and runs example.py (i.e. transcription model) to create a MIDI file from it.

In [ ]:
import subprocess, sys

note = 69
wav_file = f"note_reference_{note}.wav"
midi_file = f"note_soundfont_{note}.mid"

subprocess.run([
    sys.executable,
    "example.py",
    f"--audio_path={wav_file}",
    f"--output_midi_path={midi_file}",
], check=True)

print("Created:", midi_file)

The script below opens one MIDI file, extracts its note events, saves them to a text file, plots MIDI notes and statistics, previews the text in the notebook, and prints where the outputs were saved.

In [ ]:
import mido
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown


note = 69


midi_path = f"note_soundfont_{note}.mid"
text_path = f"note_soundfont_{note}.txt"
plot_path = f"note_soundfont_{note}.png"
title_text = midi_path


def midi_to_text_and_plot(
    midi_path,
    text_path,
    plot_path,
    title_text="MIDI Analysis",
    preview_lines=300
):
    midi = mido.MidiFile(midi_path)
    tempo = 500000
    tpq = midi.ticks_per_beat

    vel_vals, vel_times, notes, note_times = [], [], [], []

    with open(text_path, "w") as f:
        f.write(f"MIDI File: {midi_path}\n")
        f.write(f"Ticks per Beat: {tpq}\n\n")

        for i, track in enumerate(midi.tracks):
            f.write(f"Track {i}: {track.name}\n")
            f.write("-" * 40 + "\n")
            ticks = 0

            for msg in track:
                ticks += msg.time

                if msg.type == "set_tempo":
                    tempo = msg.tempo

                sec = mido.tick2second(ticks, tpq, tempo)

                if msg.type == "note_on" and msg.velocity > 0 and msg.note==note:
                    vel_vals.append(msg.velocity)
                    vel_times.append(sec)
                    notes.append(msg.note)
                    note_times.append(sec)

                    f.write(
                        f"Time {sec:.3f} sec | NOTE_ON  | "
                        f"Note: {msg.note:3d} | Velocity: {msg.velocity:3d}\n"
                    )

                elif msg.type == "note_off" or (msg.type == "note_on" and msg.velocity == 0):
                    f.write(
                        f"Time {sec:.3f} sec | NOTE_OFF | "
                        f"Note: {msg.note:3d} | Velocity: 0\n"
                    )

            f.write("\n")

    if vel_vals:
        fig, ax = plt.subplots(2, 2, figsize=(11, 7.5))

        ax[0, 0].scatter(vel_times, vel_vals, alpha=0.7)
        ax[0, 0].set_title("Velocity vs. Time")
        ax[0, 0].set_xlabel("Time (seconds)")
        ax[0, 0].set_ylabel("Velocity (0-127)")
        ax[0, 0].set_ylim(0, 127)
        ax[0, 0].set_xlim(0, 127)
        ax[0, 0].grid()
        ax[0, 0].text(-0.15, 1.12, title_text, transform=ax[0, 0].transAxes, fontsize=12)

        ax[1, 0].scatter(note_times, notes, color="purple", alpha=0.7)
        ax[1, 0].set_title("MIDI Note vs. Time")
        ax[1, 0].set_xlabel("Time (seconds)")
        ax[1, 0].set_ylabel("MIDI Note (21-108)")
        ax[1, 0].set_ylim(21, 108)
        ax[1, 0].grid()

        ax[0, 1].hist(
            vel_vals, bins=128, range=(0, 127),
            color="green", alpha=0.7, edgecolor="black"
        )
        ax[0, 1].set_title("Histogram of Velocity Values")
        ax[0, 1].set_xlabel("Velocity")
        ax[0, 1].set_ylabel("Count")
        ax[0, 1].set_xlim(0, 127)
        ax[0, 1].grid()

        ax[1, 1].hist(
            notes, bins=88, range=(21, 108),
            color="orange", alpha=0.7, edgecolor="black"
        )
        ax[1, 1].set_title("Histogram of MIDI Notes")
        ax[1, 1].set_xlabel("MIDI Note (21-108)")
        ax[1, 1].set_ylabel("Count")
        ax[1, 1].set_xlim(21, 108)
        ax[1, 1].grid()

        plt.tight_layout()
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        #display(Markdown("### Saved PNG preview"))
        #display(Image(filename=plot_path))
    else:
        display(Markdown("**No NOTE_ON events with velocity > 0 were found.**"))

    display(Markdown(f"### Text preview: `{text_path}`"))
    with open(text_path, "r") as f:
        lines = f.readlines()

    preview = "".join(lines[:preview_lines])
    if len(lines) > preview_lines:
        preview += "\n... (truncated) ..."

    display(Markdown(f"```text\n{preview}\n```"))

    print(f"PLOTS saved to: {plot_path}")
    print(f"MIDI events saved to: {text_path}")




midi_to_text_and_plot(
    midi_path=midi_path,
    text_path=text_path,
    plot_path=plot_path,
    title_text=title_text,
    preview_lines=300
)